In [1]:
import os, pickle, re
import numpy as np
import pandas as pd
import json
import hist

from physics.simulation import mcfm, msq
from physics.hstar import eft
from physics.variables import eft_terms, c6_degree, ct_degree, cg_degree
from physics.constants import c6_to_cH, cg_to_cHG, ct_to_ctH, cH_to_c6
from nsbi import carl, taylr

import torch
from torch.utils.data import TensorDataset, DataLoader
import lightning as L

In [2]:
torch.set_float32_matmul_precision('medium')

import matplotlib, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,
    "pgf.preamble": r"\usepackage{amssymb}",
})

In [3]:
run_dir_4l = 'run/h4l'
run_dir_2l2v = 'run/h2l2v'

In [4]:
(events_gg_4l_calib, events_gg_4l), scaler_X_taylr_4l, models_taylr_4l, scalers_y_taylr_4l = taylr.utils.load_results(run_dir_4l, eft_terms)
(events_gg_2l2v_calib, events_gg_2l2v), scaler_X_taylr_2l2v, models_taylr_2l2v, scalers_y_taylr_2l2v = taylr.utils.load_results(run_dir_2l2v, eft_terms)

In [5]:
(events_qq_4l_calib, events_qq_4l), _, scaler_carl_4l, model_carl_4l = carl.utils.load_results(run_dir_4l)
(events_qq_2l2v_calib, events_qq_2l2v), _, scaler_carl_2l2v, model_carl_2l2v = carl.utils.load_results(run_dir_2l2v)

In [6]:
lumi = 3000.

c6_linspace = [-25,25,251]
c6_space = np.linspace(*c6_linspace)
c6_sm = 0.0
i_c6_sm = np.where(np.isclose(c6_space,c6_sm))[0][0]
# c6_asimov = -1.0
c6_asimov = np.linspace(-10,10,11)
cH_asimov = c6_asimov * c6_to_cH
# i_c6_asimov = np.where(np.isclose(c6_space,c6_asimov))[0][0]
i_c6_asimov = np.where(np.isclose(c6_space[:, None], c6_asimov[None, :]))[0]

In [7]:
class ZeroWeightFilter():
    def __init__(self):
        pass
    def __call__(self, kinematics, components, weights, probabilities):
        return np.where(weights != 0)

sample_size = 100_000

# CALIBRATION

events_gg_4l_calib, events_gg_2l2v_calib = events_gg_4l_calib.sample(sample_size), events_gg_2l2v_calib.filter(ZeroWeightFilter()).sample(sample_size)
events_qq_4l_calib, events_qq_2l2v_calib = events_qq_4l_calib.sample(sample_size), events_qq_2l2v_calib.filter(ZeroWeightFilter()).sample(sample_size)

events_4l_calib = mcfm.stack(events_gg_4l_calib, events_qq_4l_calib)
events_2l2v_calib = mcfm.stack(events_gg_2l2v_calib, events_qq_2l2v_calib)

# TESTING

events_gg_4l, events_gg_2l2v = events_gg_4l.sample(sample_size), events_gg_2l2v.filter(ZeroWeightFilter()).sample(sample_size)
events_qq_4l, events_qq_2l2v = events_qq_4l.sample(sample_size), events_qq_2l2v.filter(ZeroWeightFilter()).sample(sample_size)

w_gg_4l_sm, w_gg_2l2v_sm = events_gg_4l.weights.to_numpy(), events_gg_2l2v.weights.to_numpy()
w_qq_4l_sm, w_qq_2l2v_sm = events_qq_4l.weights.to_numpy(), events_qq_2l2v.weights.to_numpy()

sigma_gg_4l_sm = np.sum(w_gg_4l_sm)
sigma_gg_2l2v_sm = np.sum(w_gg_2l2v_sm)
sigma_qq_4l_sm = np.sum(w_qq_4l_sm)
sigma_qq_2l2v_sm = np.sum(w_qq_2l2v_sm)

events_4l = mcfm.stack(events_gg_4l, events_qq_4l)
events_2l2v = mcfm.stack(events_gg_2l2v, events_qq_2l2v)
sigma_4l_sm = events_4l.weights.sum()
sigma_2l2v_sm = events_2l2v.weights.sum()

events_gg = mcfm.stack(events_gg_4l, events_gg_2l2v)
events_qq = mcfm.stack(events_qq_4l, events_qq_2l2v)
sigma_gg_sm = events_gg.weights.sum()
sigma_qq = events_qq.weights.sum()

In [8]:
features_4l = ['l1_pt', 'l1_eta', 'l1_phi', 'l1_energy', 'l2_pt', 'l2_eta', 'l2_phi', 'l2_energy', 'l3_pt', 'l3_eta', 'l3_phi', 'l3_energy', 'l4_pt', 'l4_eta', 'l4_phi', 'l4_energy']
features_2l2v = ["l1_pt", "l1_eta", "l1_phi", "l1_energy", "l2_pt", "l2_eta", "l2_phi", "l2_energy", "met", "met_phi"]
batch_size = 2048

In [9]:
# 4l

w_gg_4l_sm = torch.tensor(events_gg_4l.weights.to_numpy())
sigma_gg_4l_sm = torch.sum(w_gg_4l_sm)

eft_mod_gg_4l = eft.Modifier(events=events_gg_4l)

w_gg_4l_c6, _ =  torch.tensor(eft_mod_gg_4l.modify(c6_space, None, None))
sigma_gg_4l_c6 = torch.sum(w_gg_4l_c6, dim=0)  # c6 & cHbox

# 2l2v

w_gg_2l2v_sm = torch.tensor(events_gg_2l2v.weights.to_numpy())
sigma_gg_2l2v_sm = torch.sum(w_gg_2l2v_sm)

eft_mod_gg_2l2v = eft.Modifier(events=events_gg_2l2v)

w_gg_2l2v_c6, _ =  torch.tensor(eft_mod_gg_2l2v.modify(c6_space, None, None))
sigma_gg_2l2v_c6 = torch.sum(w_gg_2l2v_c6, dim=0)  # c6 & cHbox

/tmp/ipykernel_3431477/2710277505.py:8: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /opt/conda/conda-bld/pytorch_1708025847130/work/torch/csrc/utils/tensor_new.cpp:275.)
  w_gg_4l_c6, _ =  torch.tensor(eft_mod_gg_4l.modify(c6_space, None, None))


In [10]:
# # TEST: getting the ordering of events wrong should mess with everything below
# # w_pp_sm = mcfm.stack(events_gg_2l2v,events_qq_4l,events_gg_4l,events_qq_2l2v).weights.to_numpy()

w_pp_4l_sm = torch.tensor(mcfm.stack(events_gg_4l,events_qq_4l).weights.to_numpy())
sigma_pp_4l_sm = torch.sum(w_pp_4l_sm)
p_pp_4l_sm = w_pp_4l_sm / sigma_pp_4l_sm
nu_pp_4l_sm = sigma_pp_4l_sm * lumi

sigma_pp_4l_c6 = sigma_gg_4l_c6 + sigma_qq_4l_sm
nu_pp_4l_c6 = sigma_pp_4l_c6 * lumi

w_pp_2l2v_sm = torch.tensor(mcfm.stack(events_gg_2l2v,events_qq_2l2v).weights.to_numpy())
sigma_pp_2l2v_sm = torch.sum(w_pp_2l2v_sm)
p_pp_2l2v_sm = w_pp_2l2v_sm / sigma_pp_2l2v_sm
nu_pp_2l2v_sm = sigma_pp_2l2v_sm * lumi

sigma_pp_2l2v_c6 = sigma_gg_2l2v_c6 + sigma_qq_2l2v_sm
nu_pp_2l2v_c6 = sigma_pp_2l2v_c6 * lumi

In [11]:
# 4l

eft_mod_gg_4l = eft.Modifier(events=events_gg_4l)

w_gg_4l_c6_asimov, _ = eft_mod_gg_4l.modify(c6_asimov, None, None)
w_gg_4l_c6_asimov = torch.tensor(w_gg_4l_c6_asimov)

w_pp_4l_c6_asimov = torch.cat((w_gg_4l_c6_asimov, torch.tensor(events_qq_4l.weights.to_numpy())[:,None].repeat(1, w_gg_4l_c6_asimov.shape[1]) ), dim=0)
sigma_pp_4l_c6_asimov = torch.sum(w_pp_4l_c6_asimov, dim=0)
nu_pp_4l_c6_asimov = sigma_pp_4l_c6_asimov * lumi

# 2l2v

eft_mod_gg_2l2v = eft.Modifier(events=events_gg_2l2v)

w_gg_2l2v_c6_asimov, _ = eft_mod_gg_2l2v.modify(c6_asimov, None, None)
w_gg_2l2v_c6_asimov = torch.tensor(w_gg_2l2v_c6_asimov)

w_pp_2l2v_c6_asimov = torch.cat((w_gg_2l2v_c6_asimov, torch.tensor(events_qq_2l2v.weights.to_numpy())[:,None].repeat(1, w_gg_4l_c6_asimov.shape[1]) ))
sigma_pp_2l2v_c6_asimov = torch.sum(w_pp_2l2v_c6_asimov, dim=0)
nu_pp_2l2v_c6_asimov = sigma_pp_2l2v_c6_asimov * lumi

In [12]:
# Poisson term
t_4l_c6_rate = -2 * nu_pp_4l_c6_asimov[:,None] * (torch.log(nu_pp_4l_c6)[None,:] - torch.log(nu_pp_4l_sm)) + 2 * (nu_pp_4l_c6[None,:] - nu_pp_4l_sm) 
t_2l2v_c6_rate = -2 * nu_pp_2l2v_c6_asimov[:,None] * (torch.log(nu_pp_2l2v_c6)[None,:] - torch.log(nu_pp_2l2v_sm)) + 2 * (nu_pp_2l2v_c6[None,:] - nu_pp_2l2v_sm) 
t_c6_rate = t_4l_c6_rate + t_2l2v_c6_rate

# Shape (NSBI) terms

In [13]:
r_4l = carl.utils.get_likelihood_ratio(events_4l, features_4l, scaler_carl_4l, model_carl_4l)
r_2l2v = carl.utils.get_likelihood_ratio(events_2l2v, features_2l2v, scaler_carl_2l2v, model_carl_2l2v)
r = torch.cat((r_4l, r_2l2v))

coeffs_4l = taylr.utils.get_coefficients(events_4l, features_4l, scaler_X_taylr_4l, models_taylr_4l, scalers_y_taylr_4l)
coeffs_2l2v = taylr.utils.get_coefficients(events_2l2v, features_2l2v, scaler_X_taylr_2l2v, models_taylr_2l2v, scalers_y_taylr_2l2v)
coeffs = torch.cat((coeffs_4l,coeffs_2l2v),dim=0)

coeffs_gg_4l = taylr.utils.get_coefficients(events_gg_4l, features_4l, scaler_X_taylr_4l, models_taylr_4l, scalers_y_taylr_4l)
coeffs_gg_2l2v = taylr.utils.get_coefficients(events_gg_2l2v, features_2l2v, scaler_X_taylr_2l2v, models_taylr_2l2v, scalers_y_taylr_2l2v)
coeffs_gg = torch.cat((coeffs_gg_4l,coeffs_gg_2l2v),dim=0)

r_4l_calib = carl.utils.get_likelihood_ratio(events_4l_calib, features_4l, scaler_carl_4l, model_carl_4l)
r_2l2v_calib = carl.utils.get_likelihood_ratio(events_2l2v_calib, features_2l2v, scaler_carl_2l2v, model_carl_2l2v)

coeffs_4l_calib = taylr.utils.get_coefficients(events_4l_calib, features_4l, scaler_X_taylr_4l, models_taylr_4l, scalers_y_taylr_4l)
coeffs_2l2v_calib = taylr.utils.get_coefficients(events_2l2v_calib, features_2l2v, scaler_X_taylr_2l2v, models_taylr_2l2v, scalers_y_taylr_2l2v)

coeffs_gg_4l_calib = taylr.utils.get_coefficients(events_gg_4l_calib, features_4l, scaler_X_taylr_4l, models_taylr_4l, scalers_y_taylr_4l)
coeffs_gg_2l2v_calib = taylr.utils.get_coefficients(events_gg_2l2v_calib, features_2l2v, scaler_X_taylr_2l2v, models_taylr_2l2v, scalers_y_taylr_2l2v)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

...

Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: EarlyStopping, ModelCheckpoint
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
/afs/ipp-garching.mpg.de/home/t/taepa/.local/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=127` in the `DataLoader` to improve performance.


Predicting: |                                                                                                 …

In [14]:
# 4l

carl_4l = sigma_qq_4l_sm * torch.tensor(r_4l)[:,None]
taylr_4l_c6 = sigma_gg_4l_sm * torch.tensor(eft.msq_eft_over_sm(coeffs_4l, c6_space, None, None))
pratio_pp_4l_c6 =  (sigma_gg_4l_sm + sigma_qq_4l_sm)/(sigma_gg_4l_c6+sigma_qq_4l_sm) * (taylr_4l_c6 + carl_4l)/(sigma_gg_4l_sm + carl_4l)

# 2l2v

carl_2l2v = sigma_qq_2l2v_sm * torch.tensor(r_2l2v)[:,None]
taylr_2l2v_c6 = sigma_gg_2l2v_sm * torch.tensor(eft.msq_eft_over_sm(coeffs_2l2v, c6_space, None, None))
pratio_pp_2l2v_c6 =  (sigma_gg_2l2v_sm + sigma_qq_2l2v_sm)/(sigma_gg_2l2v_c6+sigma_qq_2l2v_sm) * (taylr_2l2v_c6 + carl_2l2v)/(sigma_gg_2l2v_sm + carl_2l2v)

p_pp_4l_sm = torch.tensor(p_pp_4l_sm)
pratio_pp_4l_c6 = torch.tensor(pratio_pp_4l_c6)
p_pp_4l_c6 = pratio_pp_4l_c6 * p_pp_4l_sm[:, None]
psum_pp_4l_c6 = torch.sum(p_pp_4l_c6, dim=0) 
pratio_pp_4l_c6 /= psum_pp_4l_c6[None,:]

p_pp_2l2v_sm = torch.tensor(p_pp_2l2v_sm)
pratio_pp_2l2v_c6 = torch.tensor(pratio_pp_2l2v_c6)
p_pp_2l2v_c6 = pratio_pp_2l2v_c6 * p_pp_2l2v_sm[:, None]
psum_pp_2l2v_c6 = torch.sum(p_pp_2l2v_c6, dim=0) 
pratio_pp_2l2v_c6 /= psum_pp_2l2v_c6[None,:]

/tmp/ipykernel_3431477/464565275.py:3: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  carl_4l = sigma_qq_4l_sm * torch.tensor(r_4l)[:,None]
/ptmp/mpp/taepa/higgs-offshell-interpretation/physics/hstar/eft.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  eft_coefficients = torch.tensor(eft_coefficients)
/tmp/ipykernel_3431477/464565275.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  carl_2l2v = sigma_qq_2l2v_sm * torch.tensor(r_2l2v)[:,None]
/tmp/ipykernel_3431477/464565275.py:13: UserWarning: To copy construct from a tensor, it is r

In [ ]:
def pratio_calibration(w_gg_sm, w_qq_sm, coeffs_gg, coeffs_pp, r_pp, c6_values):

    w_gg_sm = torch.tensor(w_gg_sm)
    w_qq_sm = torch.tensor(w_qq_sm)

    sigma_gg_sm = torch.sum(torch.tensor(w_gg_sm))
    sigma_qq_sm = torch.sum(torch.tensor(w_qq_sm))

    w_gg_bsm = w_gg_sm[:,None] * torch.tensor(eft.msq_eft_over_sm(coeffs_gg, c6_values, None, None))
    sigma_gg_bsm = torch.sum(w_gg_bsm, dim=0)

    w_pp_sm = torch.cat((w_gg_sm, w_qq_sm))
    p_pp_sm = w_pp_sm / torch.sum(w_pp_sm)

    carl_term = sigma_qq_sm * torch.tensor(r_pp)[:,None]
    taylr_term = sigma_gg_sm * torch.tensor(eft.msq_eft_over_sm(coeffs_pp, c6_values, None, None))
    pratio_pp_bsm = (sigma_gg_sm + sigma_qq_sm)/(sigma_gg_bsm+sigma_qq_sm) * (taylr_term + carl_term)/(sigma_gg_sm + carl_term)

    p_pp_bsm = pratio_pp_bsm * p_pp_sm[:, None]
    psum_pp_bsm = torch.sum(p_pp_bsm, dim=0) 

    poffset = (1 - psum_pp_bsm[None,:])

    return pratio_pp_bsm, poffset

In [ ]:
pratio_pp_4l_c6, poffset_pp_4l_c6 = pratio_calibration(events_gg_4l.weights, events_qq_4l.weights, coeffs_gg_4l, coeffs_4l, r_4l, c6_space)
pratio_pp_2l2v_c6, poffset_pp_2l2v_c6 = pratio_calibration(events_gg_2l2v.weights, events_qq_2l2v.weights, coeffs_gg_2l2v, coeffs_2l2v, r_2l2v, c6_space)

pratio_pp_4l_c6_calib, poffset_pp_4l_c6_calib = pratio_calibration(events_gg_4l_calib.weights, events_qq_4l_calib.weights, coeffs_gg_4l_calib, coeffs_4l_calib, r_4l_calib, c6_space)
pratio_pp_2l2v_c6_calib, poffset_pp_2l2v_c6_calib = pratio_calibration(events_gg_2l2v_calib.weights, events_qq_2l2v_calib.weights, coeffs_gg_2l2v_calib, coeffs_2l2v_calib, r_2l2v_calib, c6_space)

# pratio_pp_4l_c6 += poffset_pp_4l_c6
# pratio_pp_2l2v_c6 += poffset_pp_2l2v_c6

pratio_pp_4l_c6 += poffset_pp_4l_c6_calib
pratio_pp_2l2v_c6 += poffset_pp_2l2v_c6_calib

print(torch.sum(pratio_pp_4l_c6 * p_pp_4l_sm[:, None],dim=0))

/tmp/ipykernel_3431477/3026792927.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sigma_gg_sm = torch.sum(torch.tensor(w_gg_sm))
/tmp/ipykernel_3431477/3026792927.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sigma_qq_sm = torch.sum(torch.tensor(w_qq_sm))
/ptmp/mpp/taepa/higgs-offshell-interpretation/physics/hstar/eft.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  eft_coefficients = torch.tensor(eft_coefficients)
/tmp/ipykernel_3431477/3026792927.py:15: UserWarning: To copy construct from a tensor, it is recommended to use

tensor(-3.0058e-08, dtype=torch.float64)
tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,

In [ ]:
print(poffset_pp_2l2v_c6_calib[0,i_c6_sm])
print(poffset_pp_2l2v_c6[0,i_c6_sm])

tensor(-3.0058e-08, dtype=torch.float64)
tensor(4.2583e-09, dtype=torch.float64)
tensor(-1.4767e-07, dtype=torch.float64)
tensor(-1.2046e-07, dtype=torch.float64)


In [78]:
plt.figure()
plt.plot(c6_space, poffset_pp_4l_c6_calib[0], linestyle='-', label='calibration')
plt.plot(c6_space, poffset_pp_4l_c6[0], linestyle='--', label='asimov')
plt.legend()
plt.savefig('pcalib_pp_vs_c6.pdf')

In [72]:
t_4l_c6_shape = - 2 * torch.sum(lumi * torch.tensor(w_pp_4l_c6_asimov)[:,:,None] * torch.log(pratio_pp_4l_c6)[:,None,:] , dim=0)
t_2l2v_c6_shape = - 2 * torch.sum(lumi * torch.tensor(w_pp_2l2v_c6_asimov)[:,:,None] * torch.log(pratio_pp_2l2v_c6)[:,None,:] , dim=0)
t_c6_shape = t_4l_c6_shape + t_2l2v_c6_shape

/tmp/ipykernel_3431477/768533455.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_4l_c6_shape = - 2 * torch.sum(lumi * torch.tensor(w_pp_4l_c6_asimov)[:,:,None] * torch.log(pratio_pp_4l_c6)[:,None,:] , dim=0)
/tmp/ipykernel_3431477/768533455.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_2l2v_c6_shape = - 2 * torch.sum(lumi * torch.tensor(w_pp_2l2v_c6_asimov)[:,:,None] * torch.log(pratio_pp_2l2v_c6)[:,None,:] , dim=0)


In [73]:
t_rate = torch.tensor(t_c6_rate)
t_shape = torch.tensor(t_c6_shape)
t = t_rate + t_shape

cx_name = 'c6'
cy_name = 'ct'

cx_space = c6_asimov * c6_to_cH
cy_space = c6_space * c6_to_cH

cx_min, cx_max = -22, 22
cy_min, cy_max = -60, 80

cx_label = '$C_H$'
cy_label = '$\\hat{C}_{H}$'

/tmp/ipykernel_3431477/4034913987.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_rate = torch.tensor(t_c6_rate)
/tmp/ipykernel_3431477/4034913987.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  t_shape = torch.tensor(t_c6_shape)


In [74]:
# Find min values and their indices along axis=1 (i.e., per row)

def fit_min_max_vs_asimov(t):
    t_min_vals, t_min_indices = t.min(dim=1)  # both shape [20]

    # Get the c6_space values where t is minimum
    cH_fit_vs_asimov = (c6_space * c6_to_cH)[t_min_indices]  # shape [20]

    t_min_vals_expanded = t_min_vals[:, None]  # shape [20, 1]
    mask_1sigma = t <= (t_min_vals_expanded + 1)
    mask_2sigma = t <= (t_min_vals_expanded + 4)

    i_cH_in1sig_vs_asimov = [torch.nonzero(row_mask).squeeze(1) for row_mask in mask_1sigma]
    i_cH_in2sig_vs_asimov = [torch.nonzero(row_mask).squeeze(1) for row_mask in mask_2sigma]

    cH_in1sig_vs_asimov = [(c6_space * c6_to_cH)[indices] for indices in i_cH_in1sig_vs_asimov]
    cH_in2sig_vs_asimov = [(c6_space * c6_to_cH)[indices] for indices in i_cH_in2sig_vs_asimov]

    cH_1sig_min_vs_asimov = [c6_vals.min() for c6_vals in cH_in1sig_vs_asimov]
    cH_1sig_max_vs_asimov = [c6_vals.max() for c6_vals in cH_in1sig_vs_asimov]
    cH_1sig_interval = np.array(list(zip(cH_1sig_min_vs_asimov,cH_1sig_max_vs_asimov)))

    cH_2sig_min_vs_asimov = [c6_vals.min() for c6_vals in cH_in2sig_vs_asimov]
    cH_2sig_max_vs_asimov = [c6_vals.max() for c6_vals in cH_in2sig_vs_asimov]
    cH_2sig_interval = np.array(list(zip(cH_2sig_min_vs_asimov,cH_2sig_max_vs_asimov)))

    return cH_fit_vs_asimov, cH_1sig_interval, cH_2sig_interval

In [75]:
plt.figure(figsize=(5,4))

ax = plt.gca()

cH_fit_vs_asimov_nsbi, cH_1sigma_vs_asimov_nsbi, cH_2sigma_vs_asimov_nsbi = fit_min_max_vs_asimov(t)
cH_fit_vs_asimov_rate, cH_1sigma_vs_asimov_rate, cH_2sigma_vs_asimov_rate = fit_min_max_vs_asimov(t_rate)

print(cH_2sigma_vs_asimov_nsbi[10,0], cH_2sigma_vs_asimov_nsbi[10,1])

plt.errorbar(x=cx_space, y=cH_fit_vs_asimov_nsbi, linestyle='none', marker='+', color='blue')

plt.plot(cx_space, cH_1sigma_vs_asimov_nsbi[:,0], linestyle='--', color='blue')
plt.plot(cx_space, cH_1sigma_vs_asimov_nsbi[:,1], linestyle='--', color='blue')
plt.plot(cx_space, cH_1sigma_vs_asimov_rate[:,0], linestyle='--', color='grey')
plt.plot(cx_space, cH_1sigma_vs_asimov_rate[:,1], linestyle='--', color='grey')

plt.plot(cx_space, cH_2sigma_vs_asimov_nsbi[:,0], linestyle='-', color='blue')
plt.plot(cx_space, cH_2sigma_vs_asimov_nsbi[:,1], linestyle='-', color='blue')
plt.plot(cx_space, cH_2sigma_vs_asimov_rate[:,0], linestyle='-', color='grey')
plt.plot(cx_space, cH_2sigma_vs_asimov_rate[:,1], linestyle='-', color='grey')

# legend
rate = Line2D([0], [0], color='grey', label='$\\mathrm{Rate}$')
nsbi = Line2D([0], [0], color='blue', label='$\\mathrm{NSBI}$')
sigma1 = Line2D([0], [0], color='black', linestyle='--', label='$1\\sigma$')
sigma2 = Line2D([0], [0], color='black', linestyle='-', label='$2\\sigma$')
bestfit = Line2D([0], [0], color='black', marker='+', linestyle='None', label='$\\textrm{Best-fit}$')
empty = Line2D([0], [0], color='none', label='')
labels_hstar = [rate, nsbi, empty, sigma1, sigma2, bestfit]
leg_hstar = ax.legend(handles=labels_hstar, frameon=False, loc='upper right', ncols=2)

plt.text(0.04 ,0.96, '$pp \\rightarrow ZZ$', transform=ax.transAxes, ha='left', va='top', fontsize=12)
plt.text(0.04 ,0.88, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax.transAxes, ha='left', va='top', fontsize=12)

plt.xlabel(cx_label, fontsize=12)
plt.ylabel(cy_label, fontsize=12)

plt.xlim(cx_min, cx_max)
plt.ylim(cy_min, cy_max)

plt.tick_params(axis='both', labelsize=12)

plt.tight_layout()
plt.savefig(f'nsbi_vs_rate_{cx_name}.pdf', bbox_inches='tight')

-43.50289998506596 26.869438226070145


In [69]:
import gc
torch.cuda.empty_cache()
gc.collect()

738

In [24]:
# eft_mod_gg_4l = eft.Modifier(events=events_gg_4l)

# w_gg_4l_c6_scan, _ = eft_mod_gg_4l.modify(c6_space, None, None)
# w_gg_4l_c6_scan = torch.tensor(w_gg_4l_c6_scan)

# w_pp_4l_c6_scan = torch.cat((w_gg_4l_c6_scan, torch.tensor(events_qq_4l.weights.to_numpy())[:,None].repeat(1, w_gg_4l_c6_scan.shape[1]) ), dim=0)
# sigma_pp_4l_c6_scan = torch.sum(w_pp_4l_c6_scan, dim=0)
# nu_pp_4l_c6_scan = sigma_pp_4l_c6_scan * lumi

# # 2l2v

# eft_mod_gg_2l2v = eft.Modifier(events=events_gg_2l2v)

# w_gg_2l2v_c6_scan, _ = eft_mod_gg_2l2v.modify(c6_space, None, None)
# w_gg_2l2v_c6_scan = torch.tensor(w_gg_2l2v_c6_scan)

# w_pp_2l2v_c6_scan = torch.cat((w_gg_2l2v_c6_scan, torch.tensor(events_qq_2l2v.weights.to_numpy())[:,None].repeat(1, w_gg_4l_c6_scan.shape[1]) ), dim=0)
# sigma_pp_2l2v_c6_scan = torch.sum(w_pp_2l2v_c6_scan, dim=0)
# nu_pp_2l2v_c6_scan = sigma_pp_2l2v_c6_scan * lumi

In [25]:
# # OO analysis

# c6_opt = -10
# cH_opt = c6_opt * c6_to_cH
# i_c6_opt = np.where(np.isclose(c6_space,c6_opt))[0][0]

# c6_base = 0
# cH_base = c6_opt * c6_to_cH
# i_c6_base = np.where(np.isclose(c6_space,c6_base))[0][0]

# x_4l = pratio_pp_4l_c6[:,i_c6_opt]/pratio_pp_4l_c6[:,i_c6_base]
# x_2l2v = pratio_pp_2l2v_c6[:,i_c6_opt]/pratio_pp_2l2v_c6[:,i_c6_base]
# x_min, x_max = min(x_4l.min(), x_2l2v.min()), max(x_2l2v.max(), x_2l2v.max())

# h_c6_asimov = []
# h_c6_scan = [[] for _ in range(len(c6_asimov))]

# for i in range(len(c6_asimov)):
#     h = hist.Hist(hist.axis.Regular(20, x_min, x_max))
#     h.fill(x_4l, weight=w_pp_4l_c6_asimov[:,i])
#     h.fill(x_2l2v, weight=w_pp_2l2v_c6_asimov[:,i])
#     h_c6_asimov.append(h)

#     for j in range(len(c6_space)):
#         h = hist.Hist(hist.axis.Regular(20, x_min, x_max))
#         h.fill(x_4l, weight=w_pp_4l_c6_scan[:,j])
#         h.fill(x_2l2v, weight=w_pp_2l2v_c6_scan[:,j])
#         h_c6_scan[i].append(h)

In [26]:
# import numpy as np
# from scipy.special import gammaln  # gammaln(n+1) = log(n!)

# def poisson_nll(observed, expected, include_constant=True):
#     observed = np.asarray(observed)
#     expected = np.asarray(expected)

#     # Avoid log(0) by replacing zeros in expected with small positive value
#     expected = np.where(expected > 0, expected, 1e-20)

#     # Main NLL expression
#     nll = expected - observed * np.log(expected)

#     if include_constant:
#         nll += gammaln(observed + 1)

#     return np.sum(nll)

# t_oo = np.zeros((len(c6_asimov), len(c6_space)))
# for i in range(len(c6_asimov)):
#     h_exp = h_c6_asimov[i]
#     for j in range(len(c6_space)):
#         h_obs = h_c6_scan[i][j]
#         t_oo[i][j] = poisson_nll(h_obs.values() * lumi, h_exp.values() * lumi)

# t_min_oo = np.min(t_oo, axis=1)
# i_t_min_oo = np.argmin(t_oo, axis=1)

# print(c6_space[i_t_min_oo])

In [27]:
# plt.figure(figsize=(5,4))

# ax = plt.gca()

# cH_fit_vs_asimov_nsbi, cH_min_vs_asimov_nsbi, cH_max_vs_asimov_nsbi = fit_min_max_vs_asimov(t)
# cH_fit_vs_asimov_rate, cH_min_vs_asimov_rate, cH_max_vs_asimov_rate = fit_min_max_vs_asimov(t_rate)

# plt.errorbar(x=cx_space, y=cH_fit_vs_asimov_nsbi, linestyle='none', marker='+')

# plt.plot(cx_space, cH_min_vs_asimov_nsbi, linestyle='--', color='blue')
# plt.plot(cx_space, cH_max_vs_asimov_nsbi, linestyle='--', color='blue')

# plt.plot(cx_space, cH_min_vs_asimov_rate, linestyle='--', color='grey')
# plt.plot(cx_space, cH_max_vs_asimov_rate, linestyle='--', color='grey')

# X, Y = np.meshgrid(c6_asimov * c6_to_cH, c6_space * c6_to_cH)
# pcm = plt.contour(X, Y, (t_oo - t_min_oo[:,None]).T, levels=[-np.inf, 1], linestyles=['--'])

# labels = [
#     Line2D([0],[0],color='grey',linestyle='--',label='$\\mathrm{Rate}$'),
#     Line2D([0],[0],color='blue',linestyle='--',label='$\\mathrm{NSBI}$'),
# ]
# plt.legend(frameon=False, handles=labels, loc='upper right', ncols=2)

# plt.text(0.04 ,0.96, '$pp \\rightarrow ZZ$', transform=ax.transAxes, ha='left', va='top', fontsize=12)
# plt.text(0.04 ,0.88, '$\\sqrt{s} = 14\\,\\mathrm{TeV},\\  3\\, \\mathrm{ab}^{-1}$', transform=ax.transAxes, ha='left', va='top', fontsize=12)

# plt.xlabel(cx_label, fontsize=12)
# plt.ylabel(cy_label, fontsize=12)

# plt.xlim(cx_min, cx_max)
# plt.ylim(cy_min, cy_max)

# plt.tick_params(axis='both', labelsize=12)

# plt.tight_layout()
# plt.savefig(f'nsbi_vs_oo_{cx_name}.pdf', bbox_inches='tight')